## 1. Download Dataset from Kaggle

The retail orders dataset is downloaded from Kaggle using the Kaggle CLI.
The dataset is then extracted locally for further processing using Pandas.

In [ ]:
# Download the dataset from Kaggle
!kaggle datasets download ankitbansal06/retail-orders -f orders.csv

In [ ]:
# Extract the downloaded ZIP file
import zipfile

with zipfile.ZipFile("orders.csv.zip", "r") as zip_ref:
    zip_ref.extractall()

In [ ]:
import pandas as pd

df = pd.read_csv('orders.csv', na_values=['Not Available','unknown']) ## Treat specified placeholder values as missing values
df['Ship Mode'].unique()

In [2]:
# Standardize column names
df.columns = df.columns.str.lower()
df.columns = df.columns.str.replace(" ", "_", regex=False)

In [4]:
# Derive discount, sale price and profit
df["discount"] = df["list_price"] * df["discount_percent"] * 0.01
df["sale_price"] = df["list_price"] - df["discount"]
df["profit"] = df["sale_price"] - df["cost_price"]

In [5]:
#convert order date from object data type to datetime
df['order_date'] = pd.to_datetime(df['order_date'], format='%Y-%m-%d')


In [ ]:
# Remove source pricing columns after deriving calculated metrics
df.drop(columns=["cost_price", "list_price", "discount_percent"],inplace=True)

## 2. Connect to SQL Server

Connect to the local SQL Server database using SQLAlchemy, pyodbc, and Windows Authentication.

In [8]:
import sqlalchemy as sal
import pyodbc

engine = sal.create_engine(
    "mssql+pyodbc://localhost/PracticeDB"
    "?driver=ODBC+Driver+18+for+SQL+Server"
    "&Trusted_Connection=yes"
    "&TrustServerCertificate=yes"
)

conn = engine.connect()

print("Connected successfully!")

Connected successfully!


## 3. Load Data into SQL Server

Load the transformed Pandas DataFrame into the `df_orders` table in the `PracticeDB` database.

In [ ]:
df.to_sql('df_orders', con=conn, index=False, if_exists = 'append')